# Heterogeneous patient × valve durability generator

**Purpose:** build a semi-synthetic proof-of-concept cohort for personalized pre-operative valve durability prediction.

This notebook extends the empirical-bootstrap generator by introducing **heterogeneous patient × valve interactions**:

- **Patient trajectories:** complete longitudinal lab + medication records are bootstrapped from all 17 real patients, preserving real temporal structure, sparsity and missingness.
- **Prior valve history:** sampled from Qwen-derived valve histories in the real cohort.
- **Candidate valves:** sampled from the observed Qwen SAVR/TAVR type/model/size pool.
- **Counterfactual design:** the same synthetic patient is evaluated under multiple observed candidate-valve configurations.
- **Heterogeneous durability:** candidate effects depend on the patient's physiology, medication trajectory and prior valve history. There is intentionally **no universal best valve** in the synthetic ground truth.
- **Target:** synthetic time from hypothetical implantation at `t=0` to prosthetic failure/reintervention or censoring.

This is a **semi-synthetic development cohort**. Patient/valve inputs are grounded in the supplied real data; long-term durability outcomes and counterfactual effects are simulated and must not be presented as clinical evidence.

In [1]:

from pathlib import Path
import ast
import hashlib
import json
import math
import numpy as np
import pandas as pd

SEED = 20260917
rng = np.random.default_rng(SEED)

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------

N_BASE_PATIENTS = 1000
N_CANDIDATES_PER_PATIENT = 4

# Preserve most of the real donor pattern while creating nearby variants.
LAB_PATIENT_OFFSET_SCALE = 0.08
LAB_POINT_NOISE_SCALE = 0.05
MED_FLIP_PROB = 0.04

# Keep matched record/history donor most of the time.
# Remaining cases recombine two real donors to widen the synthetic support.
MATCH_HISTORY_PROB = 0.70

MIN_CENSOR_YEARS = 7.0
MAX_CENSOR_YEARS = 12.0

REAL_TIMELINE_CSV = Path("real_17_patient_year_lab_med_with_qwen_anchor.csv")
QWEN_HISTORY_PKL = Path("qwen_prior_valve_history_17.pkl")

OUTDIR = Path("synthetic_heterogeneous_preop_v5")
OUTDIR.mkdir(exist_ok=True)

required = [REAL_TIMELINE_CSV, QWEN_HISTORY_PKL]
missing = [str(p) for p in required if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Missing input(s):\n  - "
        + "\n  - ".join(missing)
        + "\nRun merge_all_data_PREOP_DECISION_SUPPORT.ipynb first."
    )

print("Output directory:", OUTDIR.resolve())


Output directory: /home/ioulios/dockathon/synthetic_heterogeneous_preop_v5


In [2]:

# ============================================================
# 1. Load the REAL empirical libraries
# ============================================================

real_timeline = pd.read_csv(REAL_TIMELINE_CSV)
qwen_history = pd.read_pickle(QWEN_HISTORY_PKL)

LAB_COLS = [c for c in real_timeline.columns if c.startswith("lab__")]
MED_COLS = [
    c for c in real_timeline.columns
    if c.startswith("med__") and c.endswith("__present")
]

real_timeline["Year"] = pd.to_numeric(
    real_timeline["Year"],
    errors="coerce",
)

real_timeline = (
    real_timeline
    .dropna(subset=["Patient", "Year"])
    .sort_values(["Patient", "Year"])
    .reset_index(drop=True)
)

REAL_PATIENTS = sorted(real_timeline["Patient"].dropna().unique().tolist())

print("Real patient-year table:", real_timeline.shape)
print("Real trajectory donors:", len(REAL_PATIENTS))
print("Lab channels:", len(LAB_COLS))
print("Medication channels:", len(MED_COLS))
print("Qwen history rows:", len(qwen_history))

display(
    real_timeline[
        ["Patient", "Year", "labs_observed", "medications_observed"]
        + LAB_COLS[:4]
        + MED_COLS[:4]
    ].head(10)
)


Real patient-year table: (83, 60)
Real trajectory donors: 17
Lab channels: 30
Medication channels: 15
Qwen history rows: 17


,Patient,Year,labs_observed,medications_observed,lab__Hemoglobin,lab__Hematocrit,lab__RBC,lab__WBC,med__beta_blocker__present,med__ace_inhibitor__present,med__arb__present,med__arni__present
0,Patient_101,2003,0,1,NaN,NaN,NaN,NaN,1.0,1.0,0.0,0.0
1,Patient_101,2005,1,1,12.6,35.55,3.915,8.54,0.0,0.0,0.0,0.0
2,Patient_101,2006,1,1,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
3,Patient_101,2007,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Patient_101,2008,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Patient_101,2009,1,1,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
6,Patient_101,2011,0,1,NaN,NaN,NaN,NaN,0.0,0.0,1.0,0.0
7,Patient_101,2012,0,1,NaN,NaN,NaN,NaN,0.0,1.0,0.0,0.0
8,Patient_101,2013,0,1,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
9,Patient_101,2014,0,1,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0


In [3]:

# ============================================================
# 2. Utilities + empirical lab ranges
# ============================================================

def listify(x):
    if isinstance(x, list):
        return x
    if x is None:
        return []
    if isinstance(x, float) and np.isnan(x):
        return []
    try:
        y = ast.literal_eval(str(x))
        return y if isinstance(y, list) else []
    except Exception:
        return []


def as_bool(x):
    if x is True:
        return True
    if x is False or x is None:
        return False
    return str(x).strip().lower() in {"true", "1", "yes", "y"}


def safe_float(x):
    try:
        y = float(x)
        return y if np.isfinite(y) else np.nan
    except Exception:
        return np.nan


def robust_sigma_from_series(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if len(s) == 0:
        return 1.0

    q25 = s.quantile(0.25)
    q75 = s.quantile(0.75)
    iqr = float(q75 - q25)

    if np.isfinite(iqr) and iqr > 0:
        return max(iqr / 1.349, 1e-6)

    sd = float(s.std()) if len(s) > 1 else np.nan
    if np.isfinite(sd) and sd > 0:
        return sd

    med = abs(float(s.median()))
    return max(0.10 * med, 0.10)


LAB_REFERENCE = {}

for c in LAB_COLS:
    s = pd.to_numeric(real_timeline[c], errors="coerce").dropna()

    LAB_REFERENCE[c] = {
        "median": float(s.median()) if len(s) else np.nan,
        "sigma": robust_sigma_from_series(s),
        "min": float(s.min()) if len(s) else np.nan,
        "max": float(s.max()) if len(s) else np.nan,
    }


def clip_lab(feature, value):
    ref = LAB_REFERENCE[feature]
    sigma = ref["sigma"]

    lo = ref["min"]
    hi = ref["max"]

    if np.isfinite(lo):
        lo = lo - 0.25 * sigma
    else:
        lo = -np.inf

    if np.isfinite(hi):
        hi = hi + 0.25 * sigma
    else:
        hi = np.inf

    return float(np.clip(value, lo, hi))


def stable_model_effect(model_name):
    # Tiny deterministic synthetic effect so model identity is learnable,
    # WITHOUT pretending that the observed commercial model has known
    # durability superiority in this 17-patient cohort.
    text = str(model_name or "UNKNOWN").encode("utf-8")
    h = hashlib.sha256(text).hexdigest()
    u = int(h[:8], 16) / 0xFFFFFFFF
    return (u - 0.5) * 0.12   # approximately [-0.06, +0.06]


In [4]:

# ============================================================
# 3. Build the empirical Qwen valve-procedure pool
# ============================================================

procedure_rows = []

for _, r in qwen_history.iterrows():
    patient = r.get("Patient")

    types = [str(x).upper().strip() for x in listify(r.get("procedure_types", []))]
    models = listify(r.get("valve_models", []))
    sizes = listify(r.get("valve_sizes_mm", []))
    redo = listify(r.get("redo_flags", []))
    viv = listify(r.get("ViV_flags", []))

    n = max(
        len(types),
        len(models),
        len(sizes),
        len(redo),
        len(viv),
        0,
    )

    for j in range(n):
        ptype = types[j] if j < len(types) else None

        if ptype not in {"SAVR", "TAVR"}:
            continue

        model = models[j] if j < len(models) else None
        size = safe_float(sizes[j] if j < len(sizes) else np.nan)

        # Simple plausibility guard for aortic prosthetic valve sizes.
        if np.isfinite(size) and not (18 <= size <= 35):
            size = np.nan

        procedure_rows.append({
            "source_valve_patient": patient,
            "source_procedure_index": j,
            "candidate_procedure_type": ptype,
            "candidate_valve_model": (
                str(model).strip()
                if model is not None and str(model).strip() not in {"", "None", "nan"}
                else "UNKNOWN_MODEL"
            ),
            "candidate_valve_size_mm": size,
            "candidate_valve_position": "aortic",
            "source_redo": int(as_bool(redo[j])) if j < len(redo) else 0,
            "source_ViV": int(as_bool(viv[j])) if j < len(viv) else 0,
        })

observed_procedure_pool = pd.DataFrame(procedure_rows)

if len(observed_procedure_pool) == 0:
    raise ValueError("No SAVR/TAVR procedures were recovered from the Qwen history table.")

# Collapse identical candidate configurations and retain empirical frequency.
candidate_catalog = (
    observed_procedure_pool
    .groupby(
        [
            "candidate_procedure_type",
            "candidate_valve_model",
            "candidate_valve_size_mm",
            "candidate_valve_position",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        observed_frequency=("source_valve_patient", "size"),
        n_source_patients=("source_valve_patient", "nunique"),
    )
)

candidate_catalog["candidate_id"] = [
    f"OBS_{i+1:03d}"
    for i in range(len(candidate_catalog))
]

candidate_catalog["generator_model_effect"] = (
    candidate_catalog["candidate_valve_model"]
    .map(stable_model_effect)
)

# Prefer configurations with at least a model OR size.
candidate_catalog["candidate_information_score"] = (
    (candidate_catalog["candidate_valve_model"] != "UNKNOWN_MODEL").astype(int)
    + candidate_catalog["candidate_valve_size_mm"].notna().astype(int)
)

candidate_pool = candidate_catalog[
    candidate_catalog["candidate_information_score"] >= 1
].copy()

# If the real notes are too sparse, fall back to all observed type configurations.
if candidate_pool["candidate_id"].nunique() < 2:
    candidate_pool = candidate_catalog.copy()

if candidate_pool["candidate_id"].nunique() < 2:
    raise ValueError(
        "Need at least two distinct observed Qwen valve configurations "
        "for counterfactual candidate comparisons."
    )

print("Observed procedure instances:", len(observed_procedure_pool))
print("Unique candidate configurations:", len(candidate_catalog))
print("Eligible candidate configurations:", len(candidate_pool))

display(
    candidate_pool[
        [
            "candidate_id",
            "candidate_procedure_type",
            "candidate_valve_model",
            "candidate_valve_size_mm",
            "observed_frequency",
            "n_source_patients",
        ]
    ].sort_values(
        ["candidate_procedure_type", "candidate_valve_size_mm"]
    )
)


Observed procedure instances: 19
Unique candidate configurations: 16
Eligible candidate configurations: 14


,candidate_id,candidate_procedure_type,candidate_valve_model,candidate_valve_size_mm,observed_frequency,n_source_patients
2,OBS_003,SAVR,Carpentier-Edwards bovine pericardial,21.0,1,1
1,OBS_002,SAVR,23-mm pericardial prosthesis,23.0,1,1
3,OBS_004,SAVR,Carpentier-Edwards bovine pericardial,23.0,1,1
4,OBS_005,SAVR,Carpentier-Edwards bovine pericardial aortic v...,23.0,1,1
7,OBS_008,SAVR,Perimount,23.0,1,1
8,OBS_009,SAVR,Trifecta,23.0,1,1
9,OBS_010,SAVR,Trifecta valve,23.0,1,1
11,OBS_012,SAVR,trifecta,23.0,1,1
5,OBS_006,SAVR,Carpentier-Edwards pericardial,25.0,1,1
6,OBS_007,SAVR,Carpentier-Edwards prosthetic aortic valve (si...,25.0,1,1


In [5]:

# ============================================================
# 4. Build empirical prior-valve-history donor pool
# ============================================================

history_rows = []

for _, r in qwen_history.iterrows():
    patient = r.get("Patient")

    types = [str(x).upper().strip() for x in listify(r.get("procedure_types", []))]
    sizes = pd.to_numeric(
        pd.Series(listify(r.get("valve_sizes_mm", []))),
        errors="coerce",
    ).dropna()

    models = [
        str(x).strip()
        for x in listify(r.get("valve_models", []))
        if x is not None and str(x).strip() not in {"", "None", "nan"}
    ]

    history_rows.append({
        "history_donor": patient,
        "prior_valve_count": int(sum(t in {"SAVR", "TAVR"} for t in types)),
        "prior_has_SAVR": int("SAVR" in types),
        "prior_has_TAVR": int("TAVR" in types),
        "prior_redo_count": int(sum(as_bool(x) for x in listify(r.get("redo_flags", [])))),
        "prior_ViV_count": int(sum(as_bool(x) for x in listify(r.get("ViV_flags", [])))),
        "prior_latest_valve_size_mm": float(sizes.iloc[-1]) if len(sizes) else np.nan,
        "prior_known_model_count": len(models),
    })

history_pool = pd.DataFrame(history_rows).drop_duplicates("history_donor")

PRIOR_HISTORY_FEATURES = [
    "prior_valve_count",
    "prior_has_SAVR",
    "prior_has_TAVR",
    "prior_redo_count",
    "prior_ViV_count",
    "prior_latest_valve_size_mm",
    "prior_known_model_count",
]

display(history_pool)


,history_donor,prior_valve_count,prior_has_SAVR,prior_has_TAVR,prior_redo_count,prior_ViV_count,prior_latest_valve_size_mm,prior_known_model_count
0,Patient_101,1,0,1,0,0,29.0,1
1,Patient_102,1,1,0,0,0,23.0,1
2,Patient_103,2,1,1,1,0,23.0,0
3,Patient_104,2,1,1,1,0,25.0,1
4,Patient_105,1,0,1,0,0,26.0,1
5,Patient_106,1,1,0,0,0,23.0,1
6,Patient_107,1,1,0,0,0,23.0,1
7,Patient_108,1,1,0,0,0,25.0,1
8,Patient_109,1,1,0,0,0,23.0,1
9,Patient_110,1,1,0,0,0,23.0,1


In [6]:

# ============================================================
# 5. Bootstrap COMPLETE real longitudinal trajectories
# ============================================================

real_by_patient = {
    patient: g.sort_values("Year").reset_index(drop=True)
    for patient, g in real_timeline.groupby("Patient")
}

trajectory_rows = []
synthetic_meta_rows = []

for i in range(N_BASE_PATIENTS):
    syn_patient = f"SYN_{i+1:05d}"

    record_donor = rng.choice(REAL_PATIENTS)
    donor = real_by_patient[record_donor].copy()

    # Usually preserve the real matched valve history from the same person,
    # but sometimes recombine donors to widen synthetic support.
    if (
        rng.random() < MATCH_HISTORY_PROB
        and record_donor in set(history_pool["history_donor"])
    ):
        history_donor = record_donor
    else:
        history_donor = rng.choice(history_pool["history_donor"].to_numpy())

    hrow = history_pool.loc[
        history_pool["history_donor"].eq(history_donor)
    ].iloc[0]

    # Hypothetical surgery occurs 1 year after the donor's latest recorded year.
    # Therefore EVERY donor row is strictly pre-operative.
    surgery_year = float(donor["Year"].max() + 1)

    # Persistent patient-specific offsets create nearby variants rather than
    # exact clones while retaining the donor's real longitudinal shape.
    lab_offsets = {
        c: rng.normal(0.0, LAB_PATIENT_OFFSET_SCALE * LAB_REFERENCE[c]["sigma"])
        for c in LAB_COLS
    }

    for _, row in donor.iterrows():
        out = {
            "Patient": syn_patient,
            "time_from_implant_months": float(
                (float(row["Year"]) - surgery_year) * 12.0
            ),
            "months_before_implant": float(
                (surgery_year - float(row["Year"])) * 12.0
            ),
            # generator-only provenance
            "generator_record_donor": record_donor,
            "generator_original_year": float(row["Year"]),
        }

        # Preserve REAL missingness exactly.
        for c in LAB_COLS:
            v = safe_float(row.get(c, np.nan))

            if not np.isfinite(v):
                out[c] = np.nan
                continue

            sigma = LAB_REFERENCE[c]["sigma"]
            perturbed = (
                v
                + lab_offsets[c]
                + rng.normal(0.0, LAB_POINT_NOISE_SCALE * sigma)
            )

            out[c] = clip_lab(c, perturbed)

        # Preserve real medication observation pattern; very small flip rate
        # creates nearby alternative trajectories.
        for c in MED_COLS:
            v = safe_float(row.get(c, np.nan))

            if not np.isfinite(v):
                out[c] = np.nan
                continue

            state = int(round(v))

            if rng.random() < MED_FLIP_PROB:
                state = 1 - state

            out[c] = state

        out["labs_observed"] = int(
            any(pd.notna(out[c]) for c in LAB_COLS)
        )
        out["medications_observed"] = int(
            any(pd.notna(out[c]) for c in MED_COLS)
        )

        trajectory_rows.append(out)

    meta = {
        "Patient": syn_patient,
        "generator_record_donor": record_donor,
        "generator_history_donor": history_donor,
        "generator_surgery_year": surgery_year,
    }

    for c in PRIOR_HISTORY_FEATURES:
        meta[c] = hrow[c]

    synthetic_meta_rows.append(meta)

synthetic_preop_full = pd.DataFrame(trajectory_rows)
synthetic_meta = pd.DataFrame(synthetic_meta_rows)

print("Synthetic trajectory rows:", len(synthetic_preop_full))
print("Synthetic patients:", synthetic_preop_full["Patient"].nunique())
print(
    "Median timepoints/patient:",
    synthetic_preop_full.groupby("Patient").size().median()
)

assert (synthetic_preop_full["time_from_implant_months"] < 0).all()

display(synthetic_preop_full.head(12))


Synthetic trajectory rows: 4989
Synthetic patients: 1000
Median timepoints/patient: 2.0


,Patient,time_from_implant_months,months_before_implant,generator_record_donor,generator_original_year,lab__Hemoglobin,lab__Hematocrit,lab__RBC,lab__WBC,lab__Platelets,...,med__anticoagulant__present,med__antiplatelet__present,med__statin__present,med__ccb__present,med__sglt2_inhibitor__present,med__digoxin__present,med__antiarrhythmic__present,med__insulin__present,labs_observed,medications_observed
0,SYN_00001,-12.0,12.0,Patient_109,2011.0,10.392613,33.589847,3.693701,9.983022,151.683797,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1,1
1,SYN_00002,-12.0,12.0,Patient_112,2011.0,9.934699,29.457392,3.586859,9.861930,214.923202,...,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1,1
2,SYN_00003,-96.0,96.0,Patient_102,2012.0,10.352825,31.432732,3.409809,7.385076,282.230307,...,1.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,1,1
3,SYN_00003,-84.0,84.0,Patient_102,2013.0,8.858484,28.422180,2.915435,6.843996,82.354498,...,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1,1
4,SYN_00003,-72.0,72.0,Patient_102,2014.0,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1,1
5,SYN_00003,-60.0,60.0,Patient_102,2015.0,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,1
6,SYN_00003,-48.0,48.0,Patient_102,2016.0,NaN,NaN,NaN,NaN,NaN,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1,1
7,SYN_00003,-36.0,36.0,Patient_102,2017.0,12.350661,38.850148,3.903199,9.656071,174.556614,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1,1
8,SYN_00003,-24.0,24.0,Patient_102,2018.0,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,1
9,SYN_00003,-12.0,12.0,Patient_102,2019.0,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1,1


In [7]:

# ============================================================
# 6. Derive trajectory-based synthetic risk state
# ============================================================

def last_nonmissing(g, feature):
    s = pd.to_numeric(g[feature], errors="coerce").dropna()
    return float(s.iloc[-1]) if len(s) else np.nan


def first_nonmissing(g, feature):
    s = pd.to_numeric(g[feature], errors="coerce").dropna()
    return float(s.iloc[0]) if len(s) else np.nan


def z_from_reference(feature, value):
    if not np.isfinite(value):
        return 0.0

    ref = LAB_REFERENCE[feature]
    med = ref["median"]
    sigma = max(ref["sigma"], 1e-6)

    if not np.isfinite(med):
        return 0.0

    return float((value - med) / sigma)


def latest_med(g, name):
    if name not in g.columns:
        return 0.0

    s = pd.to_numeric(g[name], errors="coerce").dropna()

    return float(s.iloc[-1]) if len(s) else 0.0


risk_rows = []

for patient, g in synthetic_preop_full.groupby("Patient"):
    g = g.sort_values("time_from_implant_months")

    # Recent physiology.
    creat = last_nonmissing(g, "lab__Creatinine") if "lab__Creatinine" in LAB_COLS else np.nan
    egfr = last_nonmissing(g, "lab__eGFR") if "lab__eGFR" in LAB_COLS else np.nan
    hb = last_nonmissing(g, "lab__Hemoglobin") if "lab__Hemoglobin" in LAB_COLS else np.nan
    lvef = last_nonmissing(g, "lab__LVEF") if "lab__LVEF" in LAB_COLS else np.nan
    nt = last_nonmissing(g, "lab__NT-proBNP") if "lab__NT-proBNP" in LAB_COLS else np.nan

    # Longitudinal change signals.
    creat_first = first_nonmissing(g, "lab__Creatinine") if "lab__Creatinine" in LAB_COLS else np.nan
    nt_first = first_nonmissing(g, "lab__NT-proBNP") if "lab__NT-proBNP" in LAB_COLS else np.nan

    creat_trend = (
        z_from_reference("lab__Creatinine", creat)
        - z_from_reference("lab__Creatinine", creat_first)
        if "lab__Creatinine" in LAB_COLS
        else 0.0
    )

    # NT-proBNP is highly skewed: use a log change.
    if np.isfinite(nt) and np.isfinite(nt_first):
        nt_trend = math.log1p(max(nt, 0.0)) - math.log1p(max(nt_first, 0.0))
    else:
        nt_trend = 0.0

    # Synthetic patient-risk function.
    # This is intentionally a generator rule, NOT a clinical causal claim.
    risk = 0.0

    if "lab__Creatinine" in LAB_COLS:
        risk += 0.20 * z_from_reference("lab__Creatinine", creat)

    if "lab__eGFR" in LAB_COLS:
        risk -= 0.18 * z_from_reference("lab__eGFR", egfr)

    if "lab__Hemoglobin" in LAB_COLS:
        risk -= 0.12 * z_from_reference("lab__Hemoglobin", hb)

    if "lab__LVEF" in LAB_COLS:
        risk -= 0.16 * z_from_reference("lab__LVEF", lvef)

    if "lab__NT-proBNP" in LAB_COLS and np.isfinite(nt):
        median_nt = LAB_REFERENCE["lab__NT-proBNP"]["median"]
        if np.isfinite(median_nt):
            risk += 0.18 * (
                math.log1p(max(nt, 0.0))
                - math.log1p(max(median_nt, 0.0))
            )

    risk += 0.10 * creat_trend
    risk += 0.06 * nt_trend

    risk += 0.10 * latest_med(g, "med__loop_diuretic__present")
    risk += 0.04 * latest_med(g, "med__anticoagulant__present")

    risk_rows.append({
        "Patient": patient,
        "generator_patient_trajectory_risk": float(risk),
        "generator_creatinine_trend": float(creat_trend),
        "generator_ntprobnp_log_trend": float(nt_trend),
    })

trajectory_risk = pd.DataFrame(risk_rows)

synthetic_patient_state = (
    synthetic_meta
    .merge(
        trajectory_risk,
        on="Patient",
        how="left",
        validate="one_to_one",
    )
)

# Add patient-shared survival noise + censoring.
synthetic_patient_state["generator_shared_survival_noise"] = rng.normal(
    0.0,
    0.16,
    size=len(synthetic_patient_state),
)

synthetic_patient_state["generator_censor_years"] = rng.uniform(
    MIN_CENSOR_YEARS,
    MAX_CENSOR_YEARS,
    size=len(synthetic_patient_state),
)

display(synthetic_patient_state.head())


,Patient,generator_record_donor,generator_history_donor,generator_surgery_year,prior_valve_count,prior_has_SAVR,prior_has_TAVR,prior_redo_count,prior_ViV_count,prior_latest_valve_size_mm,prior_known_model_count,generator_patient_trajectory_risk,generator_creatinine_trend,generator_ntprobnp_log_trend,generator_shared_survival_noise,generator_censor_years
0,SYN_00001,Patient_109,Patient_109,2012.0,1,1,0,0,0,23.0,1,0.662674,0.000000,0.000000,0.035303,10.068853
1,SYN_00002,Patient_112,Patient_112,2012.0,1,1,0,0,0,23.0,1,0.082576,0.000000,0.000000,-0.238554,9.735249
2,SYN_00003,Patient_102,Patient_102,2020.0,1,1,0,0,0,23.0,1,1.309511,2.192382,-0.270442,0.318585,11.499553
3,SYN_00004,Patient_102,Patient_102,2020.0,1,1,0,0,0,23.0,1,1.451808,2.114764,-0.123771,-0.090546,9.668428
4,SYN_00005,Patient_114,Patient_109,2014.0,1,1,0,0,0,23.0,1,-0.271349,0.000000,0.000000,-0.130095,11.395531


In [8]:
# ============================================================
# 7. Sample observed candidate valves + generate HETEROGENEOUS
#    patient × valve durability targets
# ============================================================

weights = (
    candidate_pool["observed_frequency"]
    .astype(float)
    .to_numpy()
)
weights = weights / weights.sum()

population_size_median = float(
    pd.to_numeric(
        candidate_pool["candidate_valve_size_mm"],
        errors="coerce",
    ).median()
)

# ------------------------------------------------------------------
# Build standardized phenotype features for interaction generation.
# These generator-only variables are NEVER written to model-ready data.
# ------------------------------------------------------------------

def latest_or_zero(g, feature):
    if feature not in g.columns:
        return 0.0
    s = pd.to_numeric(g[feature], errors="coerce").dropna()
    return float(s.iloc[-1]) if len(s) else 0.0


def standardized_latest(g, feature):
    if feature not in LAB_REFERENCE:
        return 0.0

    value = latest_or_zero(g, feature)
    ref = LAB_REFERENCE[feature]
    med = ref["median"]
    sigma = max(ref["sigma"], 1e-6)

    if not np.isfinite(value) or not np.isfinite(med):
        return 0.0

    return float(np.clip((value - med) / sigma, -4.0, 4.0))


def candidate_family_code(row):
    ptype = str(row["candidate_procedure_type"]).upper()
    size = safe_float(row["candidate_valve_size_mm"])

    if np.isfinite(size):
        size_centered = (size - population_size_median) / 3.0
    else:
        size_centered = 0.0

    return {
        "is_tavr": 1.0 if ptype == "TAVR" else 0.0,
        "is_savr": 1.0 if ptype == "SAVR" else 0.0,
        "size_centered": float(size_centered),
        "model_code": float(row["generator_model_effect"]) / 0.06
            if 0.06 != 0 else 0.0,
    }


scenario_rows = []

for _, p in synthetic_patient_state.iterrows():
    patient_id = p["Patient"]

    # Full pre-op longitudinal trajectory for this synthetic patient.
    g = synthetic_preop_full.loc[
        synthetic_preop_full["Patient"].eq(patient_id)
    ].sort_values("time_from_implant_months")

    # --------------------------------------------------------------
    # Heterogeneous patient phenotype
    # --------------------------------------------------------------
    renal_burden = (
        +0.55 * standardized_latest(g, "lab__Creatinine")
        -0.45 * standardized_latest(g, "lab__eGFR")
    )

    cardiac_burden = (
        -0.45 * standardized_latest(g, "lab__LVEF")
        +0.30 * standardized_latest(g, "lab__NT-proBNP")
    )

    anemia_burden = (
        -0.35 * standardized_latest(g, "lab__Hemoglobin")
    )

    medication_burden = (
        +0.30 * latest_med(g, "med__loop_diuretic__present")
        +0.16 * latest_med(g, "med__anticoagulant__present")
        +0.10 * latest_med(g, "med__antiarrhythmic__present")
        -0.08 * latest_med(g, "med__statin__present")
    )

    prior_intervention_burden = (
        +0.28 * float(p["prior_valve_count"])
        +0.24 * float(p["prior_redo_count"])
        +0.18 * float(p["prior_ViV_count"])
    )

    # Continuous phenotype dimensions that determine which candidate
    # properties are relatively favorable in the SYNTHETIC truth.
    phenotype = {
        "renal": float(np.clip(renal_burden, -3.0, 3.0)),
        "cardiac": float(np.clip(cardiac_burden, -3.0, 3.0)),
        "anemia": float(np.clip(anemia_burden, -3.0, 3.0)),
        "meds": float(np.clip(medication_burden, -2.0, 2.0)),
        "prior": float(np.clip(prior_intervention_burden, 0.0, 3.0)),
    }

    patient_main_risk = float(
        p["generator_patient_trajectory_risk"]
        + 0.16 * phenotype["renal"]
        + 0.14 * phenotype["cardiac"]
        + 0.08 * phenotype["anemia"]
        + 0.08 * phenotype["meds"]
        + 0.12 * phenotype["prior"]
    )

    # --------------------------------------------------------------
    # Draw multiple observed candidate-valve configurations for the
    # SAME patient, without replacement.
    # --------------------------------------------------------------
    n_available = len(candidate_pool)
    n_candidates = min(N_CANDIDATES_PER_PATIENT, n_available)

    chosen_idx = rng.choice(
        np.arange(n_available),
        size=n_candidates,
        replace=False,
        p=weights,
    )
    chosen = candidate_pool.iloc[chosen_idx].copy()

    # Patient-specific anatomical anchor.
    prior_size = safe_float(p["prior_latest_valve_size_mm"])
    preferred_size = prior_size if np.isfinite(prior_size) else population_size_median

    shared_noise = float(p["generator_shared_survival_noise"])
    censor_years = float(p["generator_censor_years"])

    for _, valve in chosen.iterrows():
        size = safe_float(valve["candidate_valve_size_mm"])

        if np.isfinite(size) and np.isfinite(preferred_size):
            size_mismatch = abs(size - preferred_size) / 3.0
        else:
            size_mismatch = 0.50

        cf = candidate_family_code(valve)

        # ----------------------------------------------------------
        # Candidate MAIN effect
        # Small by design: the project should NOT collapse to a
        # single globally best candidate.
        # ----------------------------------------------------------
        main_candidate_risk = (
            +0.07 * size_mismatch
            +0.035 * float(valve["generator_model_effect"])
        )

        # ----------------------------------------------------------
        # HETEROGENEOUS patient × valve interactions
        #
        # These are synthetic hypotheses for architecture validation,
        # NOT clinical claims.
        #
        # The signs are intentionally mixed so that candidate ranking
        # changes by phenotype.
        # ----------------------------------------------------------
        interaction_risk = 0.0

        # Procedure-type preference varies with phenotype.
        # Higher frailty-like burden can favor one procedure family in
        # synthetic truth, while stronger prior-intervention burden can
        # alter that balance.
        tavr_axis = (
            -0.16 * phenotype["cardiac"]
            -0.10 * phenotype["renal"]
            +0.12 * phenotype["prior"]
            +0.05 * phenotype["anemia"]
        )

        interaction_risk += cf["is_tavr"] * tavr_axis
        interaction_risk -= cf["is_savr"] * tavr_axis

        # Size effects are deliberately patient-dependent.
        # A larger candidate is not universally better or worse.
        size_axis = (
            -0.13 * phenotype["cardiac"]
            +0.11 * phenotype["renal"]
            -0.08 * phenotype["prior"]
            +0.06 * phenotype["meds"]
        )
        interaction_risk += cf["size_centered"] * size_axis

        # Model identity receives only a weak heterogeneous synthetic
        # interaction so the network can learn model embeddings without
        # turning a model name into a universal ranking.
        model_axis = (
            +0.05 * phenotype["renal"]
            -0.04 * phenotype["cardiac"]
            +0.04 * phenotype["prior"]
        )
        interaction_risk += cf["model_code"] * model_axis

        # Stronger mismatch penalty when prior interventions are present.
        interaction_risk += (
            0.07 * size_mismatch * phenotype["prior"]
        )

        candidate_total_risk = float(
            main_candidate_risk + interaction_risk
        )

        # ----------------------------------------------------------
        # Synthetic durability
        # ----------------------------------------------------------
        expected_years = 9.5 * math.exp(
            -0.30 * patient_main_risk
            -0.58 * candidate_total_risk
        )
        expected_years = float(np.clip(expected_years, 1.2, 16.0))

        # SAME shared stochastic realization across the candidate set,
        # which preserves fair paired counterfactual comparisons.
        true_event_years = expected_years * math.exp(shared_noise)
        true_event_years = float(np.clip(true_event_years, 0.75, 18.0))

        event = int(true_event_years <= censor_years)
        observed_years = min(true_event_years, censor_years)

        if event:
            p_reint = np.clip(
                0.42
                + 0.08 * size_mismatch
                + 0.05 * float(p["prior_redo_count"])
                + 0.03 * max(phenotype["prior"], 0.0),
                0.20,
                0.85,
            )

            event_type = (
                "reintervention"
                if rng.random() < p_reint
                else "prosthetic_failure"
            )
        else:
            event_type = "censored"

        scenario = {
            "Patient": patient_id,
            "counterfactual_group_id": patient_id,
            "candidate_id": valve["candidate_id"],
            "candidate_procedure_type": valve["candidate_procedure_type"],
            "candidate_valve_model": valve["candidate_valve_model"],
            "candidate_valve_size_mm": size,
            "candidate_valve_position": valve["candidate_valve_position"],
            "candidate_observed_frequency": int(valve["observed_frequency"]),
        }

        for c in PRIOR_HISTORY_FEATURES:
            scenario[c] = p[c]

        scenario.update({
            "duration_months": float(observed_years * 12.0),
            "event": event,
            "event_type": event_type,

            # Generator-only QA / provenance
            "generator_record_donor": p["generator_record_donor"],
            "generator_history_donor": p["generator_history_donor"],
            "generator_expected_durability_years": expected_years,
            "generator_true_event_time_months": float(true_event_years * 12.0),
            "generator_patient_main_risk": patient_main_risk,
            "generator_candidate_main_risk": float(main_candidate_risk),
            "generator_interaction_risk": float(interaction_risk),
            "generator_candidate_total_risk": candidate_total_risk,
            "generator_size_mismatch": float(size_mismatch),
            "generator_model_effect": float(valve["generator_model_effect"]),
            "generator_censor_years": censor_years,
            "generator_pheno_renal": phenotype["renal"],
            "generator_pheno_cardiac": phenotype["cardiac"],
            "generator_pheno_anemia": phenotype["anemia"],
            "generator_pheno_meds": phenotype["meds"],
            "generator_pheno_prior": phenotype["prior"],
        })

        scenario_rows.append(scenario)

synthetic_scenarios_full = pd.DataFrame(scenario_rows)

print("Scenario rows:", len(synthetic_scenarios_full))
print(
    "Median scenarios/patient:",
    synthetic_scenarios_full.groupby("Patient").size().median()
)
print(
    "Observed event rate:",
    round(float(synthetic_scenarios_full["event"].mean()), 3)
)
print(
    "Median observed duration (years):",
    round(float(synthetic_scenarios_full["duration_months"].median() / 12.0), 3)
)

display(synthetic_scenarios_full.head(10))

Scenario rows: 4000
Median scenarios/patient: 4.0
Observed event rate: 0.683
Median observed duration (years): 7.543


,Patient,counterfactual_group_id,candidate_id,candidate_procedure_type,candidate_valve_model,candidate_valve_size_mm,candidate_valve_position,candidate_observed_frequency,prior_valve_count,prior_has_SAVR,...,generator_interaction_risk,generator_candidate_total_risk,generator_size_mismatch,generator_model_effect,generator_censor_years,generator_pheno_renal,generator_pheno_cardiac,generator_pheno_anemia,generator_pheno_meds,generator_pheno_prior
0,SYN_00001,SYN_00001,OBS_005,SAVR,Carpentier-Edwards bovine pericardial aortic v...,23.0,aortic,1,1,1,...,0.122387,0.124087,0.000000,0.048567,10.068853,1.325290,-0.250808,0.144734,0.38,0.28
1,SYN_00001,SYN_00001,OBS_013,TAVR,Edwards-Sapien,29.0,aortic,1,1,1,...,0.427211,0.569180,2.000000,0.056231,10.068853,1.325290,-0.250808,0.144734,0.38,0.28
2,SYN_00001,SYN_00001,OBS_003,SAVR,Carpentier-Edwards bovine pericardial,21.0,aortic,1,1,1,...,0.020875,0.069352,0.666667,0.051730,10.068853,1.325290,-0.250808,0.144734,0.38,0.28
3,SYN_00001,SYN_00001,OBS_001,SAVR,23 TRIFECTA,NaN,aortic,1,1,1,...,0.095999,0.131831,0.500000,0.023752,10.068853,1.325290,-0.250808,0.144734,0.38,0.28
4,SYN_00002,SYN_00002,OBS_006,SAVR,Carpentier-Edwards pericardial,25.0,aortic,1,1,1,...,-0.090829,-0.042411,0.666667,0.050039,9.735249,-0.296185,-0.089667,0.204583,0.32,0.28
5,SYN_00002,SYN_00002,OBS_004,SAVR,Carpentier-Edwards bovine pericardial,23.0,aortic,1,1,1,...,-0.087814,-0.086003,0.000000,0.051730,9.735249,-0.296185,-0.089667,0.204583,0.32,0.28
6,SYN_00002,SYN_00002,OBS_003,SAVR,Carpentier-Edwards bovine pericardial,21.0,aortic,1,1,1,...,-0.058665,-0.010188,0.666667,0.051730,9.735249,-0.296185,-0.089667,0.204583,0.32,0.28
7,SYN_00002,SYN_00002,OBS_010,SAVR,Trifecta valve,23.0,aortic,1,1,1,...,-0.087806,-0.086725,0.000000,0.030885,9.735249,-0.296185,-0.089667,0.204583,0.32,0.28
8,SYN_00003,SYN_00003,OBS_013,TAVR,Edwards-Sapien,29.0,aortic,1,1,1,...,0.708588,0.850556,2.000000,0.056231,11.499553,2.503472,-0.485092,-0.111180,0.30,0.28
9,SYN_00003,SYN_00003,OBS_002,SAVR,23-mm pericardial prosthesis,23.0,aortic,1,1,1,...,0.225173,0.226258,0.000000,0.030999,11.499553,2.503472,-0.485092,-0.111180,0.30,0.28


In [9]:

# ============================================================
# 8. Build MODEL-READY trajectory / scenario / long tables
# ============================================================

TRAJECTORY_MODEL_COLS = [
    "Patient",
    "time_from_implant_months",
    "months_before_implant",
    "labs_observed",
    "medications_observed",
] + LAB_COLS + MED_COLS

synthetic_preop_model = (
    synthetic_preop_full[TRAJECTORY_MODEL_COLS]
    .sort_values(["Patient", "time_from_implant_months"])
    .reset_index(drop=True)
)

SCENARIO_MODEL_COLS = [
    "Patient",
    "counterfactual_group_id",
    "candidate_id",
    "candidate_procedure_type",
    "candidate_valve_model",
    "candidate_valve_size_mm",
    "candidate_valve_position",
    "candidate_observed_frequency",
] + PRIOR_HISTORY_FEATURES + [
    "duration_months",
    "event",
    "event_type",
]

synthetic_scenarios_model = (
    synthetic_scenarios_full[SCENARIO_MODEL_COLS]
    .copy()
)

synthetic_long_model = (
    synthetic_preop_model
    .merge(
        synthetic_scenarios_model,
        on="Patient",
        how="inner",
        validate="many_to_many",
    )
    .sort_values(
        ["Patient", "candidate_id", "time_from_implant_months"]
    )
    .reset_index(drop=True)
)

print("MODEL_READY trajectory rows:", len(synthetic_preop_model))
print("MODEL_READY scenario rows:", len(synthetic_scenarios_model))
print("MODEL_READY long rows:", len(synthetic_long_model))
print(
    "Unique patient-candidate scenarios:",
    synthetic_long_model[
        ["Patient", "candidate_id"]
    ].drop_duplicates().shape[0]
)

display(synthetic_long_model.head())


MODEL_READY trajectory rows: 4989
MODEL_READY scenario rows: 4000
MODEL_READY long rows: 19956
Unique patient-candidate scenarios: 4000


,Patient,time_from_implant_months,months_before_implant,labs_observed,medications_observed,lab__Hemoglobin,lab__Hematocrit,lab__RBC,lab__WBC,lab__Platelets,...,prior_valve_count,prior_has_SAVR,prior_has_TAVR,prior_redo_count,prior_ViV_count,prior_latest_valve_size_mm,prior_known_model_count,duration_months,event,event_type
0,SYN_00001,-12.0,12.0,1,1,10.392613,33.589847,3.693701,9.983022,151.683797,...,1,1,0,0,0,23.0,1,83.136488,1,prosthetic_failure
1,SYN_00001,-12.0,12.0,1,1,10.392613,33.589847,3.693701,9.983022,151.683797,...,1,1,0,0,0,23.0,1,86.204403,1,reintervention
2,SYN_00001,-12.0,12.0,1,1,10.392613,33.589847,3.693701,9.983022,151.683797,...,1,1,0,0,0,23.0,1,83.510717,1,prosthetic_failure
3,SYN_00001,-12.0,12.0,1,1,10.392613,33.589847,3.693701,9.983022,151.683797,...,1,1,0,0,0,23.0,1,64.510074,1,reintervention
4,SYN_00002,-12.0,12.0,1,1,9.934699,29.457392,3.586859,9.861930,214.923202,...,1,1,0,0,0,23.0,1,87.714990,1,reintervention


In [10]:

# ============================================================
# 9. Leakage + counterfactual QA
# ============================================================

# No generator/provenance columns may enter MODEL_READY.
for df_name, df in {
    "trajectory": synthetic_preop_model,
    "scenario": synthetic_scenarios_model,
    "long": synthetic_long_model,
}.items():

    leak_cols = [
        c for c in df.columns
        if c.startswith("generator_")
    ]

    assert leak_cols == [], (
        f"{df_name}: generator leakage detected: {leak_cols}"
    )

assert (
    synthetic_preop_model["time_from_implant_months"] < 0
).all(), "Model input contains non-preoperative observations."

# Every candidate for the same patient must receive EXACTLY the same trajectory.
grid_check = (
    synthetic_long_model
    .groupby(["Patient", "candidate_id"])["time_from_implant_months"]
    .apply(tuple)
    .reset_index(name="grid")
)

per_patient_grid_n = (
    grid_check
    .groupby("Patient")["grid"]
    .nunique()
)

assert (
    per_patient_grid_n == 1
).all(), "Candidate scenarios changed the patient's time grid."

# Check lab/med history itself is invariant across scenarios.
# Compare a compact row-wise hash of trajectory columns.
trajectory_signature_cols = [
    "time_from_implant_months"
] + LAB_COLS + MED_COLS

sig = (
    synthetic_long_model[
        ["Patient", "candidate_id"] + trajectory_signature_cols
    ]
    .fillna(-999999)
    .astype(str)
)

sig["row_signature"] = sig[trajectory_signature_cols].agg("|".join, axis=1)

scenario_signatures = (
    sig
    .groupby(["Patient", "candidate_id"])["row_signature"]
    .apply(tuple)
    .reset_index(name="trajectory_signature")
)

n_sig = (
    scenario_signatures
    .groupby("Patient")["trajectory_signature"]
    .nunique()
)

assert (
    n_sig == 1
).all(), "Candidate scenarios changed lab/med history."

print("✓ No generator-only leakage")
print("✓ All model trajectories are strictly pre-op")
print("✓ Every candidate sees the identical patient trajectory")


✓ No generator-only leakage
✓ All model trajectories are strictly pre-op
✓ Every candidate sees the identical patient trajectory


In [11]:

# ============================================================
# 10. Save
# ============================================================

trajectory_full_path = OUTDIR / "synthetic_preop_trajectory_v5_FULL.csv"
trajectory_model_path = OUTDIR / "synthetic_preop_trajectory_v5_MODEL_READY.csv"
scenario_full_path = OUTDIR / "synthetic_candidate_scenarios_v5_FULL.csv"
scenario_model_path = OUTDIR / "synthetic_candidate_scenarios_v5_MODEL_READY.csv"
long_model_path = OUTDIR / "synthetic_preop_candidate_long_v5_MODEL_READY.csv"
catalog_path = OUTDIR / "observed_qwen_candidate_catalog_v5.csv"

synthetic_preop_full.to_csv(trajectory_full_path, index=False)
synthetic_preop_model.to_csv(trajectory_model_path, index=False)
synthetic_scenarios_full.to_csv(scenario_full_path, index=False)
synthetic_scenarios_model.to_csv(scenario_model_path, index=False)
synthetic_long_model.to_csv(long_model_path, index=False)
candidate_catalog.to_csv(catalog_path, index=False)

encoder_spec = {
    "version": "heterogeneous_patient_valve_preop_v5",
    "patient_id": "Patient",
    "counterfactual_group_id": "counterfactual_group_id",
    "scenario_id": "candidate_id",

    "time_feature": "time_from_implant_months",
    "time_semantics": (
        "negative months before hypothetical candidate-valve implantation; "
        "the entire bootstrapped real donor record is pre-operative"
    ),

    "lab_features": LAB_COLS,
    "medication_features": MED_COLS,

    "prior_valve_history_features": PRIOR_HISTORY_FEATURES,

    "candidate_categorical_features": [
        "candidate_procedure_type",
        "candidate_valve_model",
        "candidate_valve_position",
    ],

    "candidate_numeric_features": [
        "candidate_valve_size_mm",
        "candidate_observed_frequency",
    ],

    "targets": [
        "duration_months",
        "event",
    ],

    "auxiliary_target_or_audit": [
        "event_type",
    ],

    "sampling_design": {
        "trajectory_source": "bootstrap complete longitudinal records from all 17 real patients",
        "prior_history_source": "Qwen valve-history donor pool from all 17",
        "candidate_source": "observed SAVR/TAVR type/model/size configurations from full Qwen procedure pool",
        "same_patient_multiple_candidates": True,
    },

    "split_rule": (
        "group by Patient/counterfactual_group_id; candidate scenarios "
        "from the same synthetic patient must never cross train/val/test"
    ),

    "heterogeneous_interactions": True,

    "clinical_interpretation": (
        "proof-of-concept synthetic event-free valve durability; "
        "not clinically validated and candidate effects are synthetic"
    ),
}

spec_path = OUTDIR / "encoder_input_spec_preop_v5.json"

with open(spec_path, "w") as f:
    json.dump(encoder_spec, f, indent=2)

print("✓", trajectory_full_path)
print("✓", trajectory_model_path)
print("✓", scenario_full_path)
print("✓", scenario_model_path)
print("✓", long_model_path)
print("✓", catalog_path)
print("✓", spec_path)


✓ synthetic_heterogeneous_preop_v5/synthetic_preop_trajectory_v5_FULL.csv
✓ synthetic_heterogeneous_preop_v5/synthetic_preop_trajectory_v5_MODEL_READY.csv
✓ synthetic_heterogeneous_preop_v5/synthetic_candidate_scenarios_v5_FULL.csv
✓ synthetic_heterogeneous_preop_v5/synthetic_candidate_scenarios_v5_MODEL_READY.csv
✓ synthetic_heterogeneous_preop_v5/synthetic_preop_candidate_long_v5_MODEL_READY.csv
✓ synthetic_heterogeneous_preop_v5/observed_qwen_candidate_catalog_v5.csv
✓ synthetic_heterogeneous_preop_v5/encoder_input_spec_preop_v5.json


In [14]:
# ============================================================
# 11. Final generator QA — FIXED
#
# Fixes:
#   1. "NAME" valve model -> "UNKNOWN_MODEL"
#   2. Uses idxmax() instead of groupby().first()
#      so candidate rows cannot get mixed across columns.
#   3. Checks candidate_id -> configuration consistency.
#   4. Handles phenotype summaries safely.
#   5. Re-saves corrected output files.
# ============================================================


# ============================================================
# A. CLEAN BAD QWEN MODEL LABELS
# ============================================================

BAD_MODEL_LABELS = {
    "NAME",
    "name",
    "Name",
    "",
    "nan",
    "None",
}


def clean_valve_model(x):

    if pd.isna(x):
        return "UNKNOWN_MODEL"

    x = str(x).strip()

    if x in BAD_MODEL_LABELS:
        return "UNKNOWN_MODEL"

    return x


# Clean every dataframe that carries candidate valve model
for df_name in [
    "candidate_catalog",
    "candidate_pool",
    "synthetic_scenarios_full",
    "synthetic_scenarios_model",
    "synthetic_long_model",
]:

    if df_name not in globals():
        continue

    df = globals()[df_name]

    if "candidate_valve_model" in df.columns:

        df["candidate_valve_model"] = (
            df["candidate_valve_model"]
            .apply(clean_valve_model)
        )


print("✓ Bad valve-model labels normalized to UNKNOWN_MODEL")


# ============================================================
# B. REAL DONOR DISTRIBUTIONS
# ============================================================

print("\nREAL record donors used:")

display(
    synthetic_meta[
        "generator_record_donor"
    ]
    .value_counts()
    .rename_axis(
        "real_record_donor"
    )
    .reset_index(
        name="n_synthetic_patients"
    )
)


print("\nQWEN history donors used:")

display(
    synthetic_meta[
        "generator_history_donor"
    ]
    .value_counts()
    .rename_axis(
        "qwen_history_donor"
    )
    .reset_index(
        name="n_synthetic_patients"
    )
)


# ============================================================
# C. CANDIDATE-LEVEL SUMMARY
# ============================================================

qa_candidate = (
    synthetic_scenarios_full
    .groupby(
        [
            "candidate_id",
            "candidate_procedure_type",
            "candidate_valve_model",
            "candidate_valve_size_mm",
        ],
        dropna=False,
    )
    .agg(
        n=(
            "Patient",
            "size",
        ),

        event_rate=(
            "event",
            "mean",
        ),

        median_observed_years=(
            "duration_months",
            lambda x:
                float(
                    np.median(x)
                ) / 12.0,
        ),

        median_expected_years=(
            "generator_expected_durability_years",
            "median",
        ),
    )
    .reset_index()
    .sort_values(
        "n",
        ascending=False,
    )
)


print(
    "\nCandidate use + "
    "synthetic target behavior:"
)

display(
    qa_candidate.head(25)
)


# ============================================================
# D. WITHIN-PATIENT COUNTERFACTUAL SPREAD
# ============================================================

patient_range = (
    synthetic_scenarios_full
    .groupby(
        "Patient"
    )[
        "generator_expected_durability_years"
    ]
    .agg(
        ["min", "max"]
    )
)


patient_range[
    "within_patient_candidate_range_years"
] = (
    patient_range["max"]
    - patient_range["min"]
)


print(
    "Overall event rate:",
    round(
        float(
            synthetic_scenarios_full[
                "event"
            ].mean()
        ),
        3,
    ),
)


print(
    "Median observed durability:",
    round(
        float(
            synthetic_scenarios_full[
                "duration_months"
            ].median()
            / 12.0
        ),
        3,
    ),
    "years",
)


print(
    "Median within-patient "
    "candidate durability range:",
    round(
        float(
            patient_range[
                "within_patient_candidate_range_years"
            ].median()
        ),
        3,
    ),
    "years",
)


print(
    "90th percentile within-patient "
    "candidate range:",
    round(
        float(
            patient_range[
                "within_patient_candidate_range_years"
            ].quantile(
                0.90
            )
        ),
        3,
    ),
    "years",
)


# ============================================================
# E. TRUE BEST CANDIDATE ROW PER PATIENT
#
# IMPORTANT:
# DO NOT use groupby().first().
#
# first() can mix values from different rows whenever
# some candidate columns contain NaN.
# ============================================================

best_idx = (
    synthetic_scenarios_full
    .groupby(
        "Patient"
    )[
        "generator_expected_durability_years"
    ]
    .idxmax()
)


best_per_patient = (
    synthetic_scenarios_full
    .loc[
        best_idx
    ]
    .copy()
    .sort_values(
        "Patient"
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n✓ Best candidate selected "
    "using complete original rows"
)


# ============================================================
# F. VERIFY CANDIDATE IDENTITIES
# ============================================================

candidate_identity_check = (
    synthetic_scenarios_full
    .groupby(
        "candidate_id"
    )
    .agg(

        n_types=(
            "candidate_procedure_type",
            "nunique",
        ),

        n_models=(
            "candidate_valve_model",
            "nunique",
        ),

        n_sizes=(
            "candidate_valve_size_mm",
            lambda x:
                x.dropna().nunique(),
        ),
    )
)


print(
    "\nCandidate ID identity check:"
)

display(
    candidate_identity_check
)


assert (
    candidate_identity_check[
        "n_types"
    ] <= 1
).all(), (
    "One candidate_id maps to "
    "multiple procedure types."
)


assert (
    candidate_identity_check[
        "n_models"
    ] <= 1
).all(), (
    "One candidate_id maps to "
    "multiple valve models."
)


assert (
    candidate_identity_check[
        "n_sizes"
    ] <= 1
).all(), (
    "One candidate_id maps to "
    "multiple known valve sizes."
)


print(
    "✓ candidate_id identity "
    "is internally consistent"
)


# ============================================================
# G. WHICH CANDIDATE IS BEST?
# ============================================================

best_counts = (
    best_per_patient
    .groupby(
        [
            "candidate_id",
            "candidate_procedure_type",
            "candidate_valve_model",
            "candidate_valve_size_mm",
        ],
        dropna=False,
    )
    .size()
    .reset_index(
        name="n_patients_best"
    )
    .sort_values(
        "n_patients_best",
        ascending=False,
    )
)


best_counts[
    "fraction_patients_best"
] = (
    best_counts[
        "n_patients_best"
    ]
    / len(
        best_per_patient
    )
)


print(
    "\nWhich candidate is best "
    "in the synthetic "
    "counterfactual truth?"
)

display(
    best_counts.head(25)
)


n_distinct_best = int(
    best_counts[
        "candidate_id"
    ].nunique()
)


top_best_fraction = float(
    best_counts[
        "fraction_patients_best"
    ].max()
)


print(
    "Distinct best candidate "
    "configurations:",
    n_distinct_best,
)


print(
    "Largest single-candidate "
    "best fraction:",
    round(
        top_best_fraction,
        3,
    ),
)


assert (
    n_distinct_best >= 2
), (
    "Heterogeneity failure: "
    "only one candidate "
    "is ever best."
)


assert (
    top_best_fraction < 0.90
), (
    "Heterogeneity too weak: "
    "one candidate dominates "
    "nearly everyone."
)


# ============================================================
# H. BEST PROCEDURE FAMILY
# ============================================================

best_family = (
    best_per_patient[
        "candidate_procedure_type"
    ]
    .value_counts(
        normalize=True
    )
    .rename_axis(
        "best_procedure_type"
    )
    .reset_index(
        name="fraction_patients"
    )
)


print(
    "\nBest procedure family "
    "by synthetic patient:"
)

display(
    best_family
)


# ============================================================
# I. PHENOTYPE-CONDITIONED HETEROGENEITY
#
# best_per_patient ALREADY contains generator_pheno_*.
# Do not merge them again.
# ============================================================

best_with_pheno = (
    best_per_patient
    .copy()
)


required_pheno_cols = [
    "generator_pheno_renal",
    "generator_pheno_cardiac",
    "generator_pheno_prior",
]


missing_pheno_cols = [
    c
    for c in required_pheno_cols
    if c not in best_with_pheno.columns
]


if missing_pheno_cols:

    raise KeyError(
        "Missing phenotype columns: "
        + ", ".join(
            missing_pheno_cols
        )
    )


# ============================================================
# J. RENAL / CARDIAC QUARTILES
# ============================================================

for pheno in [
    "generator_pheno_renal",
    "generator_pheno_cardiac",
]:

    best_with_pheno[
        pheno + "_quartile"
    ] = pd.qcut(
        best_with_pheno[
            pheno
        ],
        q=4,
        duplicates="drop",
    )


print(
    "\nBest procedure family "
    "across renal-phenotype "
    "quartiles:"
)


renal_table = pd.crosstab(

    best_with_pheno[
        "generator_pheno_renal_quartile"
    ],

    best_with_pheno[
        "candidate_procedure_type"
    ],

    normalize="index",
)


display(
    renal_table
)


print(
    "\nBest procedure family "
    "across cardiac-phenotype "
    "quartiles:"
)


cardiac_table = pd.crosstab(

    best_with_pheno[
        "generator_pheno_cardiac_quartile"
    ],

    best_with_pheno[
        "candidate_procedure_type"
    ],

    normalize="index",
)


display(
    cardiac_table
)


# ============================================================
# K. PRIOR-VALVE HISTORY
#
# prior phenotype is discrete and may have too few unique
# values for meaningful quartiles.
# Use the actual phenotype values / bins instead.
# ============================================================

prior_unique = (
    best_with_pheno[
        "generator_pheno_prior"
    ]
    .nunique(
        dropna=True
    )
)


print(
    "\nUnique prior-history "
    "phenotype values:",
    prior_unique,
)


if prior_unique >= 4:

    best_with_pheno[
        "prior_history_group"
    ] = pd.qcut(

        best_with_pheno[
            "generator_pheno_prior"
        ],

        q=4,

        duplicates="drop",
    )

else:

    best_with_pheno[
        "prior_history_group"
    ] = best_with_pheno[
        "generator_pheno_prior"
    ]


print(
    "\nBest procedure family "
    "across prior-valve-history "
    "groups:"
)


prior_table = pd.crosstab(

    best_with_pheno[
        "prior_history_group"
    ],

    best_with_pheno[
        "candidate_procedure_type"
    ],

    normalize="index",
)


display(
    prior_table
)


# ============================================================
# L. RANDOM PAIRED PATIENT AUDIT
# ============================================================

audit_patient = rng.choice(
    synthetic_scenarios_full[
        "Patient"
    ].unique()
)


print(
    "\nRandom paired candidate audit:",
    audit_patient,
)


audit_cols = [

    "Patient",

    "candidate_id",

    "candidate_procedure_type",

    "candidate_valve_model",

    "candidate_valve_size_mm",

    "duration_months",

    "event",

    "generator_expected_durability_years",

    "generator_patient_main_risk",

    "generator_candidate_main_risk",

    "generator_interaction_risk",

    "generator_candidate_total_risk",
]


audit_df = (
    synthetic_scenarios_full
    .loc[
        synthetic_scenarios_full[
            "Patient"
        ].eq(
            audit_patient
        ),
        audit_cols,
    ]
    .copy()
)


audit_df[
    "observed_years"
] = (
    audit_df[
        "duration_months"
    ]
    / 12.0
)


audit_df = (
    audit_df
    .sort_values(
        "generator_expected_durability_years",
        ascending=False,
    )
)


display(
    audit_df
)


# ============================================================
# M. RE-SAVE CLEANED MODEL DATA
#
# This ensures NAME -> UNKNOWN_MODEL also appears
# in the actual files used downstream.
# ============================================================

if "scenario_full_path" in globals():

    synthetic_scenarios_full.to_csv(
        scenario_full_path,
        index=False,
    )


if "scenario_model_path" in globals():

    synthetic_scenarios_model.to_csv(
        scenario_model_path,
        index=False,
    )


if "long_model_path" in globals():

    synthetic_long_model.to_csv(
        long_model_path,
        index=False,
    )


if "catalog_path" in globals():

    candidate_catalog.to_csv(
        catalog_path,
        index=False,
    )


print(
    "\n✓ Cleaned output files re-saved"
)


# ============================================================
# FINAL STATUS
# ============================================================

print(
    "\n"
    "============================================"
)

print(
    "✓ Heterogeneous patient × valve "
    "generator passed QA"
)

print(
    "✓ Complete-row best-candidate "
    "selection confirmed"
)

print(
    "✓ NAME normalized to UNKNOWN_MODEL"
)

print(
    "============================================"
)

✓ Bad valve-model labels normalized to UNKNOWN_MODEL

REAL record donors used:


,real_record_donor,n_synthetic_patients
0,Patient_102,77
1,Patient_115,70
2,Patient_105,70
3,Patient_117,67
4,Patient_110,66
5,Patient_116,62
6,Patient_107,59
7,Patient_101,58
8,Patient_112,56
9,Patient_114,56



QWEN history donors used:


,qwen_history_donor,n_synthetic_patients
0,Patient_102,83
1,Patient_101,68
2,Patient_115,67
3,Patient_105,66
4,Patient_103,64
5,Patient_117,64
6,Patient_112,62
7,Patient_116,61
8,Patient_110,58
9,Patient_107,57



Candidate use + synthetic target behavior:


...


Unique prior-history phenotype values: 2

Best procedure family across prior-valve-history groups:


candidate_procedure_type,SAVR,TAVR
prior_history_group,,
0.28,0.768707,0.231293
0.80,0.686441,0.313559



Random paired candidate audit: SYN_00732


,Patient,candidate_id,candidate_procedure_type,candidate_valve_model,candidate_valve_size_mm,duration_months,event,generator_expected_durability_years,generator_patient_main_risk,generator_candidate_main_risk,generator_interaction_risk,generator_candidate_total_risk,observed_years
2927,SYN_00732,OBS_004,SAVR,Carpentier-Edwards bovine pericardial,23.0,86.984545,0,10.387655,-0.027904,0.001811,-0.141388,-0.139578,7.248712
2925,SYN_00732,OBS_010,SAVR,Trifecta valve,23.0,86.984545,0,10.323049,-0.027904,0.001081,-0.129902,-0.128821,7.248712
2926,SYN_00732,OBS_009,SAVR,Trifecta,23.0,86.984545,0,10.223048,-0.027904,-0.000057,-0.111980,-0.112037,7.248712
2924,SYN_00732,OBS_003,SAVR,Carpentier-Edwards bovine pericardial,21.0,86.984545,0,9.650937,-0.027904,0.048477,-0.061222,-0.012745,7.248712



✓ Cleaned output files re-saved

✓ Heterogeneous patient × valve generator passed QA
✓ Complete-row best-candidate selection confirmed
✓ NAME normalized to UNKNOWN_MODEL


## What this generator is testing

The network trained on these outputs should **not** merely learn a global ranking of valves.

It must learn:

\[
S(t \mid X_i, V_j)
\]

where \(X_i\) is the patient's complete pre-operative history and \(V_j\) is the candidate valve.

Because the generator creates patient-specific interactions, the counterfactual ordering

\[
V_A > V_B > V_C
\]

can change from one patient to another. The downstream model should therefore be evaluated on both:

1. ordinary survival discrimination/calibration; and
2. **counterfactual ranking recovery** — whether it correctly identifies which candidate has higher synthetic durability for the same held-out patient.

The second metric is essential for the intended surgical decision-support demonstration.